In [3]:
import pandas as pd
import numpy as np

# =========================
# Step 2 — Train/Test splits
#   (1) Random row split
#   (2) Unseen-load split (hold out some load levels)
# =========================

# ----- Load data (assumes you already ran Step 1 and renamed columns) -----
df = pd.read_csv(r"C:\Users\NFU_Power_Lab\Desktop\DAT_ResearchArea\BestAwardAI2026\WPT.csv").rename(columns={"種類":"type","負載":"load","效率":"efficiency"})
df["type"] = df["type"].astype(str)
df["load"] = pd.to_numeric(df["load"], errors="coerce")
df["efficiency"] = pd.to_numeric(df["efficiency"], errors="coerce")
df = df.dropna(subset=["type","load","efficiency"]).reset_index(drop=True)

SEED = 42
TEST_RATIO = 0.1  # change to 0.3 if you want larger test set

rng = np.random.default_rng(SEED)

# =========================
# Split 2.1 — Random row split
# =========================
n = len(df)
idx = np.arange(n)
rng.shuffle(idx)

test_size = int(round(TEST_RATIO * n))
test_idx = idx[:test_size]
train_idx = idx[test_size:]

train_random = df.iloc[train_idx].reset_index(drop=True)
test_random  = df.iloc[test_idx].reset_index(drop=True)

print("=== Random Row Split ===")
print("Train:", train_random.shape, "Test:", test_random.shape)
print("Train types:", train_random["type"].nunique(), "Test types:", test_random["type"].nunique())
print("Train loads:", train_random["load"].nunique(), "Test loads:", test_random["load"].nunique())

# Save (optional)
train_random.to_csv("wpt_train_random.csv", index=False, encoding="utf-8-sig")
test_random.to_csv("wpt_test_random.csv", index=False, encoding="utf-8-sig")
print("Saved: wpt_train_random.csv, wpt_test_random.csv")

# =========================
# Split 2.2 — Unseen-load split
# Goal: test uses load levels NOT seen in training
# A simple, robust strategy: hold out every k-th load level
# =========================
all_loads = np.array(sorted(df["load"].unique()))

# Strategy: hold out loads at even indices (or change to mod pattern)
# This usually gives ~50% test; we want ~20%, so we pick a pattern.
# We'll pick 1 out of every 5 load levels for test (~20%).
k = 5
test_loads = set(all_loads[::k])  # 0th, 5th, 10th, ...
train_loads = set(all_loads) - test_loads

train_unseen_load = df[df["load"].isin(train_loads)].reset_index(drop=True)
test_unseen_load  = df[df["load"].isin(test_loads)].reset_index(drop=True)

print("\n=== Unseen-Load Split ===")
print("Held-out loads (test):", sorted(test_loads))
print("Train loads count:", len(train_loads), "Test loads count:", len(test_loads))
print("Train:", train_unseen_load.shape, "Test:", test_unseen_load.shape)

# Sanity checks (no load overlap)
overlap = set(train_unseen_load["load"].unique()).intersection(set(test_unseen_load["load"].unique()))
print("Load overlap (should be empty):", overlap)

# Check that all types still exist in both (usually true here)
print("Train types:", train_unseen_load["type"].nunique(), "Test types:", test_unseen_load["type"].nunique())

# Save (optional)
train_unseen_load.to_csv("wpt_train_unseen_load.csv", index=False, encoding="utf-8-sig")
test_unseen_load.to_csv("wpt_test_unseen_load.csv", index=False, encoding="utf-8-sig")
print("Saved: wpt_train_unseen_load.csv, wpt_test_unseen_load.csv")

# =========================
# Optional — Report condition coverage
# Useful to show in paper: how many (type, load) conditions in train/test
# =========================
cond_train = train_unseen_load.groupby(["type","load"]).size().reset_index(name="n")
cond_test  = test_unseen_load.groupby(["type","load"]).size().reset_index(name="n")

print("\nCondition coverage (unseen-load split):")
print("Train conditions:", cond_train.shape[0], "Test conditions:", cond_test.shape[0])
print("Min/Max repeats in train:", cond_train["n"].min(), cond_train["n"].max())
print("Min/Max repeats in test :", cond_test["n"].min(), cond_test["n"].max())

=== Random Row Split ===
Train: (1310, 3) Test: (146, 3)
Train types: 11 Test types: 10
Train loads: 26 Test loads: 26
Saved: wpt_train_random.csv, wpt_test_random.csv

=== Unseen-Load Split ===
Held-out loads (test): [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
Train loads count: 20 Test loads count: 6
Train: (1120, 3) Test: (336, 3)
Load overlap (should be empty): set()
Train types: 11 Test types: 11
Saved: wpt_train_unseen_load.csv, wpt_test_unseen_load.csv

Condition coverage (unseen-load split):
Train conditions: 220 Test conditions: 66
Min/Max repeats in train: 1 16
Min/Max repeats in test : 1 16
